In [1]:
import vectorbt as vbt
import yfinance as yf
print("vectorbt version:", getattr(vbt, "__version__", "unknown"))
print("yfinance version:", getattr(yf, "__version__", "unknown"))
print("Kernel python:", __import__("sys").executable)
import pandas as pd
import numpy as np
from datetime import datetime

vectorbt version: 0.28.5
yfinance version: 1.4.1
Kernel python: /opt/homebrew/Caskroom/miniforge/base/envs/vbt_env/bin/python


### **CONFIG**

In [2]:
# ================== CONFIG ==================
SYMBOL = "QTUM"          # or "QTUM" ETF
VIX_SYMBOL = "^VIX"
START_DATE = "2018-01-01"   # QTUM inception ~2018
END_DATE = datetime.now().strftime("%Y-%m-%d")

# MA periods
FAST_MA = 50
SLOW_MA = 200

# VIX thresholds (test multiple in sweep)
VIX_ENTER = 20
VIX_EXIT = 20   # or 25 for stricter regime filter

# Risk controls (your style)
INITIAL_CASH = 100_000
FEES = 0.001      # 0.1% round-trip approx for ETF
SLIPPAGE = 0.001

### **DATA LOADING**

In [3]:
# ================== DATA LOADING ==================

print("Downloading data...")

# Download symbols *separately* (one at a time). The multi-symbol path
# in vbt.YFData.download([list]) raises "Symbols have mismatching columns"
# with current yfinance (always emits Ticker level in columns) + vectorbt.
price = vbt.YFData.download(
    SYMBOL,
    start=START_DATE,
    end=END_DATE,
    interval="1d"
).get("Close")

vix = vbt.YFData.download(
    VIX_SYMBOL,
    start=START_DATE,
    end=END_DATE,
    interval="1d"
).get("Close")

# yfinance .history() often returns tz-aware indexes with different
# sub-day times (04:00 vs 06:00) even for the same UTC day across symbols.
# Normalize to midnight so the subsequent align() can match dates.
price.index = price.index.normalize()
vix.index = vix.index.normalize()

# Align dates
price, vix = price.align(vix, join='inner')




In [4]:
print("=== GROK TEST UPDATE SUCCESSFUL ===")
print("This print statement was added via the update tool to test notebook editing access.")
from datetime import datetime as _dt
print("Executed at:", _dt.now().isoformat())

=== GROK TEST UPDATE SUCCESSFUL ===
This print statement was added via the update tool to test notebook editing access.
Executed at: 2026-06-13T22:17:12.663523


### **INDICATORS**

In [ ]:
# ================== INDICATORS ==================
# Moving Averages
ma_fast = vbt.MA.run(price, window=FAST_MA, ewm=False)
ma_slow = vbt.MA.run(price, window=SLOW_MA, ewm=False)

# Golden / Death Cross signals
golden_cross = ma_fast.ma_crossed_above(ma_slow) 
death_cross = ma_fast.ma_crossed_below(ma_slow)

# VIX Regime
in_calm_regime = vix < VIX_ENTER
in_risk_off = vix >= VIX_EXIT

### **ENTRY / EXIT SIGNALS**

In [6]:
# ================== ENTRY / EXIT SIGNALS ==================
# Long when: Golden Cross AND calm regime
# (You can also use sustained condition: ma_fast.ma > ma_slow AND calm)
entries = golden_cross & in_calm_regime

# Exit when: Death Cross OR risk-off
exits = death_cross | in_risk_off

# Optional: Stay in position only while trend + regime both favorable
# trend_up = ma_fast.ma > ma_slow.ma
# entries = trend_up & in_calm_regime
# exits = ~trend_up | in_risk_off

### **BACKTEST**

In [7]:
# ================== BACKTEST ==================
pf = vbt.Portfolio.from_signals(
    price,
    entries=entries,
    exits=exits,
    init_cash=INITIAL_CASH,
    fees=FEES,
    slippage=SLIPPAGE,
    direction="longonly",
    freq="1D"
)

### **RESULTS**

In [8]:
# ================== RESULTS ==================
print(pf.stats())
pf.plot().show()                    # Equity curve, drawdowns, trades
# pf.orders.plot()                  # Trade visualization
# pf.drawdown.plot()                # Drawdown analysis

Start                         2018-09-05 00:00:00+00:00
End                           2026-06-12 00:00:00+00:00
Period                               1953 days 00:00:00
Start Value                                    100000.0
End Value                                  97957.375274
Total Return [%]                              -2.042625
Benchmark Return [%]                         595.061409
Max Gross Exposure [%]                            100.0
Total Fees Paid                              197.955531
Max Drawdown [%]                               2.409769
Max Drawdown Duration                 843 days 00:00:00
Total Trades                                          1
Total Closed Trades                                   1
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                        0.0
Best Trade [%]                                -2.044667
Worst Trade [%]                               -2